In [ ]:
import os
at_colab = 'COLAB_GPU' in os.environ

if at_colab:
    %pip install pygadm
    %pip install owslib
    %pip install distinctipy
    %pip install folium
    %pip install mapclassify
    %pip install selenium webdriver-manager
    %pip install wget

In [ ]:
import os
import wget

url = 'https://raw.githubusercontent.com/gromicho/tools/main/jg_folium.py'
filename = 'jg_folium.py'

# Ensure the file is removed before downloading
if os.path.exists(filename):
    os.remove(filename)

# Download the file (this will now always download the latest version)
wget.download(url, filename)


In [ ]:
import jg_folium as jg

In [ ]:
import geopandas as gpd
import pygadm
import distinctipy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

In [ ]:
country_name = 'Netherlands'
ad_level = 2

# Download the shapefile data
gdf = pygadm.Items(name=country_name, content_level=ad_level)

print(gdf.crs)

In [ ]:
gdf = gdf[gdf.ENGTYPE_2 == 'Municipality'].rename(
    columns={'NAME_1': 'Province', 'NAME_2': 'Municipality'}
).sort_values(['Province','Municipality'])

In [ ]:
def plot_municipalities(
    gdf,
    color_by: str | None = None,
    legend: str | None = None,
    ncol: int = 1,
    figsize: tuple[int, int] = (8, 6),
    title: str | None = None,
    output_file: str | None = None,
    basemap_source=None,
    basemap_crs: int | None = 3857,
) -> None:
    """
    Plot a GeoDataFrame of municipalities with optional coloring and basemap.

    Parameters
    ----------
    gdf : GeoDataFrame
        Geospatial data to plot.
    color_by : str | None, default None
        Column used for coloring (expects color values).
    legend : str | None, default None
        Column used for legend labels.
    ncol : int, default 1
        Number of legend columns.
    figsize : tuple[int, int], default (8, 6)
        Figure size.
    title : str | None, default None
        Plot title.
    output_file : str | None, default None
        If provided, saves figure to file.
    basemap_source : optional, default None
        Contextily tile source.
    basemap_crs : int | None, default 3857
        CRS to project to before adding basemap.
        If None, no reprojection is performed.
    """
    import matplotlib.pyplot as plt
    import matplotlib.lines as mlines

    fig, ax = plt.subplots(figsize=figsize)

    gdf_plot = gdf

    # ---- CRS handling ----
    if basemap_source is not None and basemap_crs is not None:
        gdf_plot = gdf.to_crs(epsg=basemap_crs)

    # ---- plotting ----
    if color_by and color_by in gdf_plot.columns:
        gdf_plot.plot(
            ax=ax,
            color=gdf_plot[color_by],
            edgecolor='black'
        )

        if legend and legend in gdf_plot.columns:
            seen = set()
            handles = []

            for c, l in zip(gdf_plot[color_by], gdf_plot[legend]):
                if (c, l) not in seen:
                    seen.add((c, l))
                    handles.append(
                        mlines.Line2D(
                            [0], [0],
                            marker='s',
                            color='w',
                            markerfacecolor=c,
                            markersize=8,
                            label=l,
                        )
                    )

            ax.legend(
                handles=handles,
                loc='upper left',
                bbox_to_anchor=(1.05, 1),
                ncol=ncol,
            )
    else:
        gdf_plot.plot(ax=ax, color='lightblue', edgecolor='black')

    # ---- basemap ----
    if basemap_source is not None:
        import contextily as ctx
        ctx.add_basemap(ax, source=basemap_source)

    # ---- cosmetics ----
    ax.set_axis_off()

    if title:
        ax.set_title(title)

    if output_file:
        plt.savefig(output_file, bbox_inches='tight', dpi=300)

    plt.show()

In [ ]:
def plot_municipalities(
    gdf,
    color_by=None,
    legend=None,
    ncol=1,
    figsize=(8, 6),
    title=None,
    output_file=None
):
    """
    Plots a GeoDataFrame of municipalities with optional coloring using a custom colormap.

    Parameters:
    - gdf (GeoDataFrame): The geospatial data to plot.
    - color_by (str, optional): Column name in gdf to use for coloring.
    - legend (str, optional): Column name for the legend labels.
    - ncol (int, optional): Number of columns in the legend. Default is 1.
    - figsize (tuple, optional): Figure size, default is (8,6).
    - title (str, optional): Custom title for the plot.
    - output_file (str, optional): File name to save the plot. If None, does not save.
    """
    fig, ax = plt.subplots(figsize=figsize)

    if color_by and color_by in gdf.columns:
        # Plot using the custom colormap
        gdf.plot(ax=ax, color=gdf[color_by], edgecolor="black")

        if legend and legend in gdf.columns:
            # Manually create a legend
            patches = [
                mpatches.Patch(color=color, label=municipality)
                for color, municipality in zip(gdf[color_by], gdf[legend])
            ]

            import matplotlib.lines as mlines

            patches = [
                mlines.Line2D(
                    [0], [0], marker='s', color='w', markersize=8,  # Adjust size
                    markerfacecolor=color, label=municipality
                )
                for color, municipality in zip(gdf[color_by], gdf[legend])
            ]

            ax.legend(
                handles=patches,
                loc="upper left",
                bbox_to_anchor=(1.05, 1),
                ncol=ncol
            )

    else:
        gdf.plot(ax=ax, edgecolor="black", color="lightblue")  # Default color

    # Set title if provided
    if title:
        ax.set_title(title)

    # Save file if output_file is provided
    if output_file:
        plt.savefig(output_file, format="pdf", bbox_inches="tight", dpi=300)

    plt.show()


In [ ]:
plot_municipalities(gdf,output_file='GADM_no_crs.pdf',basemap_crs=None)

In [ ]:
plot_municipalities(gdf,output_file='GADM_no_crs.pdf',basemap_crs=4326)

In [ ]:
plot_municipalities(gdf.set_crs(epsg=4326),output_file='GADM_crs_4326.pdf')

In [ ]:
plot_municipalities(gdf.set_crs(epsg=3857),output_file='GADM_crs_3857.pdf')

In [ ]:
plot_municipalities(gdf.set_crs(epsg=4326).to_crs(epsg=28992),output_file='GADM_crs_28992.pdf')

In [ ]:
city_names = sorted(set(gdf.Municipality))
city_colors = distinctipy.get_colors(len(city_names))

gdf['color_by_city'] = gdf['Municipality'].map(
    {
        name: mcolors.to_hex(color)
        for color, name in zip(city_colors, city_names)
    }
)

In [ ]:
if not gdf.crs:
    # Define original CRS
    gdf.set_crs(epsg=4326, inplace=True, allow_override=True)
# Reproject to EPSG:28992
gdf = gdf.to_crs(epsg=28992)

In [ ]:
plot_municipalities(
    gdf.sort_values('Municipality'),
    color_by='color_by_city',
    legend='Municipality',
    figsize=(17,19),
    ncol=5,
    output_file='GADM_colored.pdf'
)

In [ ]:
gdf_plain = gpd.GeoDataFrame(gdf.copy(), geometry=gdf.geometry, crs=gdf.crs)

In [ ]:
import contextily as ctx
ax = gdf_plain.to_crs(epsg=3857).plot(figsize=(8, 8), alpha=0.7)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
ax.set_axis_off()

In [ ]:
import geopandas as gpd
import xyzservices.providers as xyz

gdf_plot = gpd.GeoDataFrame(gdf_plain.copy(), geometry='geometry', crs=gdf_plain.crs)
gdf_plot = gdf_plot.to_crs(epsg=4326)

m = gdf_plot.explore(
    color=gdf_plot['color_by_city'],
    legend=False,
    tiles=xyz.CartoDB.Positron,
    tooltip=['Municipality', 'Province'],
    zoom_on_bounds=True,
    style_kwds={'color': 'black', 'weight': 1},
)

m.save('netherlands_map.html')
m

In [ ]:
map_all_cities = gdf_plain.explore(
    color=gdf['color_by_city'],
    legend=False,
    tiles='OpenStreetMap',
    tooltip=['Municipality','Province'],
    zoom_on_bounds=True,
    style_kwds={"color": "black", "weight": 1}, 
)
jg.FoliumToPng( map_all_cities, 'map_all_cities_gadm' )
map_all_cities

In [ ]:
map_all_provinces = gdf_plain.explore(
    column='Province',
    cmap='Paired',
    legend=True,
    categorical=True,
    tiles='OpenStreetMap',
    tooltip=['Municipality', 'Province'],
    zoom_on_bounds=True,
    style_kwds={"color": "black", "weight": 1}
)
jg.FoliumToPng( map_all_provinces, 'map_all_provinces_gadm' )
map_all_provinces

In [ ]:
map_capelle = gdf_plain[gdf_plain.Municipality.str.contains('Capelle')].explore(
    tooltip=False,
    style_kwds={'color': 'red', 'weight': 5, 'fill': False}
)
jg.FoliumToPng( map_capelle, 'map_capelle_gadm' )
map_capelle

In [ ]:
if at_colab:
    from google.colab import files
    files.download('map_all_cities_gadm.png')
    files.download('map_all_provinces_gadm.png')
    files.download('map_capelle_gadm.png')
else:
    import pyperclip
    pyperclip.copy('\r\n'.join(sorted(gdf.Municipality.values)))

In [ ]:
gdf.shape

In [ ]:
gdf.to_parquet('gadm.parquet')